In [4]:
import pandas as pd
import sqlite3

# Load the CSV
df = pd.read_csv('Technical Support Dataset.csv')

# Create the database
conn = sqlite3.connect('support_tickets.db')
df.to_sql('raw_tickets', conn, if_exists='replace', index=False)
conn.close()

print('Done! Rows loaded:', len(df))

Done! Rows loaded: 2330


In [5]:
conn = sqlite3.connect('support_tickets.db')
cur = conn.cursor()

cur.executescript('''
    DROP TABLE IF EXISTS tickets;

    CREATE TABLE tickets AS
    SELECT
        [Ticket ID]                             AS ticket_id,
        TRIM([Status])                          AS status,
        TRIM([Priority])                        AS priority,

        CASE
            WHEN LOWER(TRIM([Topic])) = 'pricing and licensing'
            THEN 'Pricing and Licensing'
            ELSE TRIM([Topic])
        END                                     AS topic,

        TRIM([Source])                          AS source,
        TRIM([Agent Group])                     AS agent_group,
        TRIM([Agent Name])                      AS agent_name,
        TRIM([Support Level])                   AS support_level,
        TRIM([Product group])                   AS product_group,
        TRIM([Country])                         AS country,

        SUBSTR([Created time], 1, 19)           AS created_time,
        SUBSTR([Resolution time], 1, 19)        AS resolution_time,
        SUBSTR([Close time], 1, 19)             AS close_time,
        SUBSTR([First response time], 1, 19)    AS first_response_time,

        TRIM([SLA For Resolution])              AS sla_resolution,
        TRIM([SLA For first response])          AS sla_first_response,

        CASE WHEN TRIM([SLA For Resolution])     = 'Within SLA' THEN 1 ELSE 0 END AS sla_met,
        CASE WHEN TRIM([SLA For first response]) = 'Within SLA' THEN 1 ELSE 0 END AS first_response_sla_met,

        [Agent interactions]                    AS agent_interactions,
        [Survey results]                        AS survey_score,
        [Latitude]                              AS latitude,
        [Longitude]                             AS longitude

    FROM raw_tickets;
''')

conn.commit()
conn.close()
print('Cleaning done!')

Cleaning done!


In [6]:
conn = sqlite3.connect('support_tickets.db')

df_clean = pd.read_sql_query('SELECT * FROM tickets LIMIT 5', conn)
print(df_clean.columns.tolist())
print()
print(df_clean.head())

conn.close()

['ticket_id', 'status', 'priority', 'topic', 'source', 'agent_group', 'agent_name', 'support_level', 'product_group', 'country', 'created_time', 'resolution_time', 'close_time', 'first_response_time', 'sla_resolution', 'sla_first_response', 'sla_met', 'first_response_sla_met', 'agent_interactions', 'survey_score', 'latitude', 'longitude']

   ticket_id       status priority                     topic source  \
0       1012       Closed      Low           Feature request  Email   
1       1013  In progress     High             Product setup  Phone   
2       1014     Resolved      Low  Purchasing and invoicing   Chat   
3       1015       Closed   Medium     Pricing and Licensing  Email   
4       1016       Closed      Low             Product setup  Email   

        agent_group           agent_name support_level  \
0  1st line support      Kristos Westoll        Tier 1   
1  2nd line support   Adolpho Messingham        Tier 2   
2  1st line support      Bernard Beckley        Tier 1   

In [7]:
conn = sqlite3.connect('support_tickets.db')

df_clean = pd.read_sql_query('SELECT * FROM tickets', conn)
df_clean.to_csv('tickets_clean.csv', index=False)

conn.close()
print('Exported! Rows:', len(df_clean))

Exported! Rows: 2330
